# GPU vs CPU Benchmark for Whisper

This notebook benchmarks OpenAI Whisper transcription performance on GPU vs CPU to verify GPU acceleration is working correctly.

In [ ]:
# Cell 1: Environment Check
# Check if CUDA is available and display GPU information

import torch
import whisper
import time

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU detected - benchmark will only test CPU performance")

In [ ]:
# Cell 2: Load Whisper Model
# Load the "small" model (can change to tiny, base, medium, large as needed)

AUDIO_FILE = "../data/US_DebateAudio.wav" 
MODEL_SIZE = "small"

print(f"Loading Whisper '{MODEL_SIZE}' model...")
model = whisper.load_model(MODEL_SIZE)
print("Model loaded successfully")

In [ ]:
# Cell 3: CPU Transcription Benchmark
# Run transcription on CPU and measure time

print("Running CPU benchmark...")
model = model.cpu()
torch.cuda.empty_cache()

start_time = time.time()
cpu_result = whisper.transcribe(model, AUDIO_FILE)
cpu_time = time.time() - start_time

print(f"CPU Transcription completed in {cpu_time:.2f} seconds")
print(f"Detected language: {cpu_result['language']}")
print(f"Text preview: {cpu_result['text'][:100]}...")

In [ ]:
# Cell 4: GPU Transcription Benchmark
# Run transcription on GPU and measure time (if GPU available)

if torch.cuda.is_available():
    print("Running GPU benchmark...")
    model = model.to("cuda")
    torch.cuda.empty_cache()
    
    start_time = time.time()
    gpu_result = whisper.transcribe(model, AUDIO_FILE)
    gpu_time = time.time() - start_time
    
    print(f"GPU Transcription completed in {gpu_time:.2f} seconds")
    print(f"Detected language: {gpu_result['language']}")
    print(f"Text preview: {gpu_result['text'][:100]}...")
else:
    print("GPU not available - skipping GPU benchmark")
    gpu_time = None

In [ ]:
# Cell 5: Side-by-Side Comparison
# Display performance comparison and speedup

print("\n" + "="*50)
print("BENCHMARK RESULTS")
print("="*50)
print(f"CPU Time: {cpu_time:.2f} seconds")

if gpu_time is not None:
    print(f"GPU Time: {gpu_time:.2f} seconds")
    speedup = cpu_time / gpu_time
    print(f"Speedup: {speedup:.2f}x faster on GPU")
    print("\n" + "="*50)
    print("PERFORMANCE INTERPRETATION")
    print("="*50)
    if speedup > 3:
        print("Excellent GPU acceleration!")
    elif speedup > 1.5:
        print("Good GPU acceleration - GPU is providing meaningful speedup")
    elif speedup > 1:
        print("Moderate GPU acceleration - consider checking CUDA configuration")
    else:
        print("Warning: GPU slower than CPU - check CUDA drivers and configuration")
else:
    print("GPU benchmark not available")
    print("\nTo enable GPU acceleration:")
    print("1. Install CUDA toolkit")
    print("2. Install PyTorch with CUDA support")
    print("3. Restart this notebook")